# DMOSpeech 2: Stage-Wise Trajectory Action Cascade (`STAC`) & `1D-CRC-STAC` on Both Grids (`16-Grid Steps 1..8` & `32-Grid Steps 1..15`)

**Evaluates the complete `STAC` and `1D-CRC-STAC` method, matched controls, ablations, OOD corruptions, and Hindsight Oracle across BOTH teacher grids (`16-interval` and `32-interval`) without any binary failure rule:**

1. **Proposed Method — `STAC` (`Stage-Wise Trajectory Action Cascade`) & `1D-CRC-STAC`:**
   - **Cumulative Trajectory Action (Zero External Overhead, No SNR Estimator):**
     $$A_k = \sum_{j=1}^{k} d_j = \sum_{j=1}^{k} \frac{1}{L_{\text{gen}}} \left\lVert \left(\hat{x}_1(x_{t_j}, t_j) - \hat{x}_1(x_{t_{j-1}}, t_{j-1})\right)_{[L_p : L]} \right\rVert_F$$
   - **Stage-Wise Stopping Cascade:** Hand over at the earliest candidate step $k \in \mathcal{K}_{\text{cand}}$ where $A_k \ge \hat{\lambda}_k$, else fall back to $k_{\text{ref}}$.
   - **1-Parameter Conformal Risk-Controlled Variant (`1D-CRC-STAC`):** Indexes $\lambda_k(\tau) = Q_{1-\tau}(\{A_k(x_i)\}_{i \in \mathcal{D}_{\text{cal}}})$ by a single scalar $\tau \in [0, 1]$ with monotone envelope loss $L_{\text{mono}}(x; \tau) = \max_{\tau' \le \tau} L(x, \pi_{\tau'}(x)) \in [0, 1]$, guaranteeing $\mathbb{E}[L_{\text{mono}}(X_{\text{test}}; \hat{\tau}_{\text{CRC}})] \le \alpha = 0.05$.

2. **Matched Controls, Ablations, OOD & Oracle Analyses Included:**
   - **`Control: Stage-Wise SNR Cascade`** ($\hat{S} \ge \hat{\theta}_k$, matched stage-wise degrees of freedom)
   - **`Ablation: Stage-Wise Single-Step Drift`** ($d_k \ge \hat{\lambda}_k$, instantaneous drift without cumulative path summation)
   - **`Ablation: Single-Threshold Trajectory Action`** ($A_k \ge \hat{\lambda}$, scalar threshold across all steps)
   - **Zero-Shot OOD Generalization (`EXP-011`, $n=100$):** Calibrated on Clean+Gaussian, evaluated zero-shot on `babble20`, `babble15`, `reverb03`, `reverb09`, and `telephone` with **Speaker-Cluster Bootstrap 95% CIs**.
   - **Hindsight Per-Utterance Oracle Sensitivity:** Evaluated across `Strict` ($\Delta\text{WER} \le 0.00, \Delta\text{SIM} \ge -0.010$), `Primary` ($\le 0.01, \ge -0.015$), and `Relaxed` ($\le 0.02, \ge -0.020$) regimes.
   - **500-Speaker Benchmark (`Full d_1..d_8`):** Reports **Raw WER**, **Capped WER ($\le 1.0$)**, and **Outlier-Trimmed WER** with Speaker-Cluster Bootstrap 95% CIs.

3. **Full Checkpoint Synthesis on Both Grids (`16-Grid Steps 1..8` & `32-Grid Steps 1..15`):**
   - **16-Grid ($t_k = 1 - \cos(\pi k / 32)$):** Reuses verified T4 scores for `{4, 5, 6, 8}` and synthesizes missing `{1, 2, 3, 7}` on the Tesla T4 GPU (`g16_all8` and `g16_le025`).
   - **32-Grid ($t_i = 1 - \cos(\pi i / 64)$):** Reuses verified T4 scores for `{7, 9, 11, 13, 14}` and synthesizes missing `{1, 2, 3, 4, 5, 6, 8, 10, 12, 15}` on the Tesla T4 GPU (`g32_all14` and `g32_all15`).

4. **Comprehensive Computational Cost, Speed ($\times$), Wall-Clock Latency, RTF & Memory Benchmarking (Section 6 & Step 6B):**
   - **Computational Cost:** Teacher ODE Steps ($k$), Teacher Function Evals ($k+1$), Teacher CFG Forward Passes ($2(k+1)$), Student Steps ($3$), Total $\text{NFE} = 2(k+1)+3$, $\%$ NFE Saved, Gate Overhead (`0.12 ms` for `STAC`/`1D-CRC-STAC` vs `1.85 ms` for SNR-only and `1.97 ms` for hybrid SNR+drift), and Peak GPU Memory (`MB`).
   - **Speed & Time:** Teacher ODE Wall-Clock Time (`s`), Teacher Speedup ($\times$), End-to-End Synthesis Wall-Clock Time (`s`), Wall-Clock Time Saved (`s` and $\%$), End-to-End Speedup ($\times = T_{\text{ref}} / T_{\text{policy}}$), Real-Time Factor ($\text{RTF} = T_{\text{synth}} / T_{\text{audio}}$), and Generation Speed ($\times\text{Real-Time} = 1 / \text{RTF}$).

### Step 1: Check Runtime (T4 GPU for Full Run: Run Steps 1–7; CPU or GPU for Instant Analysis: Run Steps 2A–2B)

In [ ]:
import torch
if torch.cuda.is_available():
    !nvidia-smi
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU — Steps 2A-2B (Instant T4 Benchmark Analysis for STAC & 1D-CRC-STAC) work on CPU! Switch to T4 GPU for Steps 3-6B.')

### Step 2A: Clone Thesis Repository (`rhnl-3` branch) & Install Analysis Packages

In [ ]:
import os, subprocess
THESIS_BRANCH = 'rhnl-3'
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)
!git -C /content/Final-Thesis-Setup branch --show-current && git -C /content/Final-Thesis-Setup log --oneline -1
!pip install -q scipy numpy pandas matplotlib
runner_path = '/content/Final-Thesis-Setup/code/colab/run_swsr_handover_prelim.py'
assert os.path.exists(runner_path), f'Missing {runner_path} on branch {THESIS_BRANCH}'
print('Branch and runner verified:', THESIS_BRANCH, runner_path)

### Step 2B: Instant Evaluation of `STAC`, `1D-CRC-STAC`, Matched Controls, OOD (`EXP-011`), & Oracle Sensitivity (`<15 seconds`)

In [ ]:
import os, subprocess, sys
from IPython.display import Markdown, display

if not os.path.exists('/content/Final-Thesis-Setup/code/colab/run_swsr_handover_prelim.py'):
    subprocess.run(['git', 'clone', '-b', 'rhnl-3', 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)

OUT_DIR = '/content/stac_benchmark_results'
os.makedirs(OUT_DIR, exist_ok=True)
subprocess.run([sys.executable, '/content/Final-Thesis-Setup/code/colab/run_swsr_handover_prelim.py', 'analyze', '--out', OUT_DIR], check=True)

with open(f'{OUT_DIR}/SWSR_HANDOVER_REPORT.md', encoding='utf-8') as f:
    display(Markdown(f.read()))

### Step 3: Clone `Shonget-DMOSpeech2` & `Final-Thesis-Setup` + Download Checkpoints (Tesla T4 Only)

In [ ]:
import os, subprocess, hashlib, torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
assert 'T4' in name, f'Steps 3-6B require a Tesla T4 GPU (found {name}). Switch runtime to T4 GPU.'

THESIS_BRANCH = 'rhnl-3'
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}

!pip install -q "numpy==2.2.3" "pandas==2.2.3" audioop-lts
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub

!mkdir -p ckpts
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_85000.pt -O ckpts/model_85000.pt
!wget -c https://huggingface.co/yl4579/DMOSpeech2/resolve/main/model_1500.pt -O ckpts/model_1500.pt

for fname, digest in {
    'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
    'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'
}.items():
    h = hashlib.sha256()
    with open(f'ckpts/{fname}', 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    assert h.hexdigest() == digest, f'{fname}: SHA-256 mismatch'
    print(f'{fname}: SHA-256 verified')

### Step 4: Download Seed-TTS `test-en` Audio

In [ ]:
import os
os.chdir('/content/Shonget-DMOSpeech2')
test_file = 'data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav'
if not os.path.exists(test_file) or os.path.getsize(test_file) < 1000:
    !rm -rf data/seed-tts-eval
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir='data/seed-tts-eval')
!ls -lh data/seed-tts-eval/en/prompt-wavs/common_voice_en_10933823.wav

### Step 5: Smoke Test on Both Grids (`16-Grid Steps 1..8` & `32-Grid Steps 1..15` + Live `STAC` Early-Exit)

In [ ]:
import os, subprocess, sys, pandas as pd
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

# Self-healing check: ensure Final-Thesis-Setup (rhnl-3) is cloned & up-to-date even if Step 2A was skipped
THESIS_BRANCH = 'rhnl-3'
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
os.chdir('/content/Shonget-DMOSpeech2')
OUT = '/content/drive/MyDrive/dmospeech2_results/swsr_steps1to8'
os.makedirs(OUT, exist_ok=True)
subprocess.run([
    sys.executable, '/content/Final-Thesis-Setup/code/colab/run_swsr_handover_prelim.py',
    'smoke', '--variant', 'stac', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2'
], check=True)

sm_df = pd.read_csv(f'{OUT}/smoke/steps1to8.csv')
display(sm_df[['utt_id', 'condition', 'prompt_snr', 't_prepare', 't_eval', 't_student', 't_vocoder', 'audio_seconds', 'peak_mem_mb', 'wer_g16_1', 'wer_g16_4', 'wer_g16_7', 'wer_g16_8', 'wer_g32_1', 'wer_g32_7', 'wer_g32_14', 'wer_g32_15']])
live_sm = pd.read_csv(f'{OUT}/smoke/live_swsr_t4.csv')
display(live_sm[['utt_id', 'condition', 'variant', 'chosen_step', 'teacher_evals_swsr', 'teacher_passes_swsr', 'nfe_swsr', 'nfe_saving_pct', 'gate_overhead_ms', 't_teacher_swsr_s', 't_teacher_ref_s', 'teacher_speedup_x', 'time_swsr_s', 'time_ref_s', 'time_saved_s', 'wall_clock_saving', 'speedup_x', 'rtf_swsr', 'rtf_ref', 'x_realtime_swsr', 'x_realtime_ref', 'peak_mem_mb', 'endpoint_max_abs_diff']])

### Step 6A: Full Synthesis Run on Both Grids (`16-Grid Steps 1..8` & `32-Grid Steps 1..15`, Resumable to Google Drive)
- By default (`SKIP_G32_ALL = False`), synthesizes all missing steps on **both** the 16-grid (`{1, 2, 3, 7}`) and the 32-grid (`{1, 2, 3, 4, 5, 6, 8, 10, 12, 15}`), merges them with the stored T4 scores from `EXP-017` (`{4, 5, 6, 8}` on 16-grid and `{7, 9, 11, 13, 14}` on 32-grid), and evaluates `STAC`, `1D-CRC-STAC`, and all matched controls across all 6 grid configurations (`g16`, `g32`, `g16_all8`, `g16_le025`, `g32_all14`, `g32_all15`).
- Set `SKIP_G32_ALL = True` if you want the faster 16-grid-only synthesis (`{1, 2, 3, 7}`, ~1.5 hours on T4).

In [ ]:
SKIP_G32_ALL = False   # False = synthesize missing steps on BOTH 16-grid (1..8) and 32-grid (1..15)

import os, subprocess, sys
from IPython.display import Markdown, display
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

THESIS_BRANCH = 'rhnl-3'
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
os.chdir('/content/Shonget-DMOSpeech2')
OUT = '/content/drive/MyDrive/dmospeech2_results/swsr_steps1to8'
os.makedirs(OUT, exist_ok=True)
cmd = [sys.executable, '/content/Final-Thesis-Setup/code/colab/run_swsr_handover_prelim.py', 'run_steps1to8', '--variant', 'stac', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2']
if SKIP_G32_ALL:
    cmd.append('--skip_g32_all')
subprocess.run(cmd, check=True)

with open(f'{OUT}/SWSR_HANDOVER_REPORT.md', encoding='utf-8') as f:
    display(Markdown(f.read()))

### Step 6B: Live Tesla T4 Early-Stopping Speed, Latency, RTF & Computational Cost Benchmark (`live_t4` with `--variant stac`)
- Executes the actual live teacher ODE loop on the Tesla T4 GPU with `STAC` early exit ($A_k = \sum_{j=1}^k d_j \ge \hat{\lambda}_k$, zero external SNR estimator) and verifies exact bitwise endpoint identity (`max|diff| = 0.0`), computational cost (`Teacher Evals`, `Teacher CFG Passes`, `Total NFE`, `% NFE Saved`, `Gate Overhead ms`, `Peak GPU Memory MB`), and speed/time (`Teacher ODE Time s`, `Teacher Speedup x`, `End-to-End Time s`, `Time Saved s & %`, `End-to-End Speedup x`, `RTF`, and `x-Real-Time Speed`).

In [ ]:
import os, subprocess, sys, pandas as pd
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

THESIS_BRANCH = 'rhnl-3'
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, 'https://github.com/Shonget-Thesis/Final-Thesis-Setup.git', '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
os.chdir('/content/Shonget-DMOSpeech2')
OUT = '/content/drive/MyDrive/dmospeech2_results/swsr_steps1to8'
os.makedirs(OUT, exist_ok=True)
subprocess.run([
    sys.executable, '/content/Final-Thesis-Setup/code/colab/run_swsr_handover_prelim.py',
    'live_t4', '--variant', 'stac', '--out', OUT, '--dmo_root', '/content/Shonget-DMOSpeech2'
], check=True)

live_df = pd.read_csv(f'{OUT}/live_swsr_t4.csv')
summary = pd.DataFrame([{
    'Policy': 'STAC (Live Early-Stop)',
    'Mean Step k': round(live_df['chosen_step'].mean(), 2),
    'Teacher Evals (k+1)': round(live_df['teacher_evals_swsr'].mean(), 2),
    'Teacher CFG Passes': round(live_df['teacher_passes_swsr'].mean(), 2),
    'Total NFE': round(live_df['nfe_swsr'].mean(), 2),
    '% NFE Saved': f"{live_df['nfe_saving_pct'].mean():.1f}%",
    'Gate Overhead (ms)': round(live_df['gate_overhead_ms'].mean(), 2),
    'Teacher Time (s)': round(live_df['t_teacher_swsr_s'].mean(), 4),
    'Teacher Speedup (x)': f"{live_df['t_teacher_ref_s'].mean() / max(live_df['t_teacher_swsr_s'].mean(), 1e-9):.2f}x",
    'End-to-End Time (s)': round(live_df['time_swsr_s'].mean(), 4),
    'Time Saved (s)': round(live_df['time_saved_s'].mean(), 4),
    '% Wall-Clock Saved': f"{live_df['wall_clock_saving'].mean() * 100:.1f}%",
    'End-to-End Speedup (x)': f"{live_df['time_ref_s'].mean() / max(live_df['time_swsr_s'].mean(), 1e-9):.2f}x",
    'RTF': round(live_df['rtf_swsr'].mean(), 4),
    'x-Real-Time Speed': f"{1.0 / max(live_df['rtf_swsr'].mean(), 1e-9):.2f}x RT",
    'Peak GPU Mem (MB)': round(live_df['peak_mem_mb'].max(), 1),
    'Max |Endpoint Diff|': live_df['endpoint_max_abs_diff'].max(),
}, {
    'Policy': 'Reference Step 8 (Full Run)',
    'Mean Step k': 8.0,
    'Teacher Evals (k+1)': 9.0,
    'Teacher CFG Passes': 18.0,
    'Total NFE': 21.0,
    '% NFE Saved': '0.0%',
    'Gate Overhead (ms)': 0.0,
    'Teacher Time (s)': round(live_df['t_teacher_ref_s'].mean(), 4),
    'Teacher Speedup (x)': '1.00x',
    'End-to-End Time (s)': round(live_df['time_ref_s'].mean(), 4),
    'Time Saved (s)': 0.0,
    '% Wall-Clock Saved': '0.0%',
    'End-to-End Speedup (x)': '1.00x',
    'RTF': round(live_df['rtf_ref'].mean(), 4),
    'x-Real-Time Speed': f"{1.0 / max(live_df['rtf_ref'].mean(), 1e-9):.2f}x RT",
    'Peak GPU Mem (MB)': round(live_df['peak_mem_mb'].max(), 1),
    'Max |Endpoint Diff|': 0.0,
}])
display(summary)
display(live_df)

### Step 7: Package & Download All Results (`STAC_Steps1to8_BothGrids_Results.zip`)

In [ ]:
!zip -r /content/STAC_Steps1to8_BothGrids_Results.zip /content/drive/MyDrive/dmospeech2_results/swsr_steps1to8
from google.colab import files
files.download('/content/STAC_Steps1to8_BothGrids_Results.zip')